In [ ]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
from src.data_loaders import *
from torch_geometric.nn import Node2Vec
from tqdm.notebook import tqdm
from sklearn.metrics import roc_auc_score
from torch.optim.lr_scheduler import ReduceLROnPlateau
import os 
import random


from IPython.display import display

sns.set_style('whitegrid')


In [19]:
movies_df, ratings_df = load_movielens_data()

print('Movies Dataset:')
display(movies_df.head())

print('Ratings Dataset:')
display(ratings_df.head())


Movies Dataset:


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


Ratings Dataset:


,userId,movieId,rating,timestamp
0,1,296,5.0,1147880044
1,1,306,3.5,1147868817
2,1,307,5.0,1147868828
3,1,665,5.0,1147878820
4,1,899,3.5,1147868510


In [20]:
movielens_graph = transform_movielenses_to_graph(ratings_df.copy(), movies_df.copy())

print(f"Number of user nodes: {len(movielens_graph['user_nodes'])}")
print(f"Number of item nodes: {len(movielens_graph['item_nodes'])}")
print(f"Number of genre nodes: {len(movielens_graph['genre_nodes'])}")
print(f"Number of user-item edges: {len(movielens_graph['user_item_edges'])}")
print(f"Number of item-genre edges: {len(movielens_graph['item_genre_edges'])}")


Number of user nodes: 162342
Number of item nodes: 62423
Number of genre nodes: 20
Number of user-item edges: 12452811
Number of item-genre edges: 112307


In [21]:
reviews_df_steam_cleaned, items_df_steam_cleaned = load_steam_data()

print("Cleaned Reviews Dataset:")
display(reviews_df_steam_cleaned.head())
print("Cleaned Items Dataset:")
display(items_df_steam_cleaned.head())


Cleaned Reviews Dataset:


,user_id,app_id
0,76561197970982479,1250
1,76561197970982479,22200
4,js41637,227300
5,js41637,239030
6,evcentric,248820


Cleaned Items Dataset:


,app_id,title,genres
0,761140,Lost Summoner Kitty,"[Action, Casual, Indie, Simulation, Strategy]"
1,643980,Ironbound,"[Free to Play, Indie, RPG, Strategy]"
2,670290,Real Pool 3D - Poolians,"[Casual, Free to Play, Indie, Simulation, Sports]"
3,767400,弹炸人2222,"[Action, Adventure, Casual]"
5,772540,Battle Royale Trainer,"[Action, Adventure, Simulation]"


In [22]:
steam_graph = transform_steam_to_graph(reviews_df_steam_cleaned, items_df_steam_cleaned)

print(f"Number of user nodes: {len(steam_graph['user_nodes'])}")
print(f"Number of item nodes: {len(steam_graph['item_nodes'])}")
print(f"Number of genre nodes: {len(steam_graph['genre_nodes'])}")
print(f"Number of user-item edges: {len(steam_graph['user_item_edges'])}")
print(f"Number of item-genre edges: {len(steam_graph['item_genre_edges'])}")


Number of user nodes: 22077
Number of item nodes: 2802
Number of genre nodes: 23
Number of user-item edges: 45261
Number of item-genre edges: 72786


In [23]:
merged_graph = merge_graphs(movielens_graph, steam_graph)

print(f"Number of user nodes in merged graph: {len(merged_graph['user_nodes'])}")
print(f"Number of item nodes in merged graph: {len(merged_graph['item_nodes'])}")
print(f"Number of genre nodes in merged graph: {len(merged_graph['genre_nodes'])}")
print(f"Number of user-item edges in merged graph: {len(merged_graph['user_item_edges'])}")
print(f"Number of item-genre edges in merged graph: {len(merged_graph['item_genre_edges'])}")


Number of user nodes in merged graph: 184419
Number of item nodes in merged graph: 65225
Number of genre nodes in merged graph: 41
Number of user-item edges in merged graph: 12497401
Number of item-genre edges in merged graph: 185092


In [24]:

# Create a DataFrame of user-item interactions
all_interactions_df = pd.DataFrame(merged_graph['user_item_edges'], columns=['user_id', 'item_id'])

# Create user and item maps
unique_users = all_interactions_df['user_id'].unique()
unique_items = all_interactions_df['item_id'].unique()
user_map = {user: i for i, user in enumerate(unique_users)}
item_map = {item: i for i, item in enumerate(unique_items)}
all_interactions_df['user_idx'] = all_interactions_df['user_id'].map(user_map)
all_interactions_df['item_idx'] = all_interactions_df['item_id'].map(item_map)

# Split the data
def split_data(interactions, test_size=0.2):
    test_indices = np.random.choice(interactions.index, size=int(len(interactions) * test_size), replace=False)
    test = interactions.loc[test_indices]
    train = interactions.drop(test_indices)
    return train, test

train_interactions_df, test_interactions_df = split_data(all_interactions_df)

# Create a training graph from string IDs
train_graph = {
    'user_nodes': merged_graph['user_nodes'],
    'item_nodes': merged_graph['item_nodes'],
    'genre_nodes': merged_graph['genre_nodes'],
    'user_item_edges': [tuple(x) for x in train_interactions_df[['user_id', 'item_id']].to_numpy()],
    'item_genre_edges': merged_graph['item_genre_edges'],
}

print(f"Number of training user-item edges: {len(train_graph['user_item_edges'])}")
print(f"Number of testing user-item edges: {len(test_interactions_df)}")


Number of training user-item edges: 9997921
Number of testing user-item edges: 2499480


In [25]:
G_pyg_train, node_to_int_id, int_id_to_node = create_pyg_graph(train_graph)
print(G_pyg_train)


Data(edge_index=[2, 20234272], num_nodes=249685)


In [26]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


In [27]:
model_path = "models/node2vec_model.pth"

if os.path.exists(model_path):
    print(f"Loading pre-trained model from {model_path}...")
    model = Node2Vec(
        edge_index=G_pyg_train.edge_index,
        embedding_dim=64,
        walk_length=20,
        context_size=10,
        walks_per_node=20,
        num_negative_samples=1,
        p=1,
        q=0.5,
        sparse=True,
    ).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    embeddings = model.embedding.weight
    print("Model loaded and embeddings are ready.")
else:
    print("No pre-trained model found. Initializing and training a new model...")
    model = Node2Vec(
        edge_index=G_pyg_train.edge_index,
        embedding_dim=64,
        walk_length=20,
        context_size=10,
        walks_per_node=20,
        num_negative_samples=1,
        p=1,
        q=0.5,
        sparse=True,
    ).to(device)

    print("Model initialization and walk generation complete.")

    print("Creating data loader...")
    loader = model.loader(batch_size=128, shuffle=True, num_workers=4)
    optimizer = torch.optim.SparseAdam(model.parameters(), lr=0.01)

    lr_scheduler = ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=0.1,
        patience=3, 
    )

    early_stopping_patience = 5
    epochs_no_improve = 0
    best_loss = float('inf')  
    max_epochs = 101 

    print("Starting training...")
    for epoch in range(1, max_epochs):
        model.train()
        total_loss = 0
        for pos_rw, neg_rw in loader:
            optimizer.zero_grad()
            loss = model.loss(pos_rw.to(device), neg_rw.to(device))
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(loader)
        current_lr = optimizer.param_groups[0]['lr']
        print(f'Epoch: {epoch}, Loss: {avg_loss:.4f}, LR: {current_lr}')

        lr_scheduler.step(avg_loss)

        if avg_loss < best_loss:
            best_loss = avg_loss
            epochs_no_improve = 0
            date = pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')
            torch.save(model.state_dict(), f'models/best_model_{epoch}_{date}.pth')
        else:
            epochs_no_improve += 1

        if epochs_no_improve == early_stopping_patience:
            print(f"Early stopping triggered after {epoch} epochs.")
            break 

    model.eval()
    embeddings = model.embedding.weight
    print("Training complete. Embeddings are ready.")


Loading pre-trained model from models/node2vec_model.pth...
Model loaded and embeddings are ready.


In [28]:
def get_recommendation_score(user_id, item_id, embeddings, mapping):
    try:
        u_id = mapping[user_id]
        v_id = mapping[item_id]

        u_emb = embeddings[u_id].cpu().detach()
        v_emb = embeddings[v_id].cpu().detach()

        return torch.dot(u_emb, v_emb).item()

    except KeyError:
        return "Node not in mapping"
    except Exception as e:
        return f"Error: {e}"

real_edges_to_test = [
    merged_graph['user_item_edges'][i]
    for i in np.random.choice(len(merged_graph['user_item_edges']), 5)
]

print("--- Testing REAL Edges ---")
for user, item in real_edges_to_test:
    score = get_recommendation_score(user, item, embeddings, node_to_int_id)
    print(f"Score for ({user}, {item}): {score:.4f}")

user_nodes = merged_graph['user_nodes']
item_nodes = merged_graph['item_nodes']
all_real_edges_set = set(merged_graph['user_item_edges'])

fake_edges_to_test = []
while len(fake_edges_to_test) < 5:
    rand_user = np.random.choice(user_nodes)
    rand_item = np.random.choice(item_nodes)

    if (rand_user, rand_item) not in all_real_edges_set:
        fake_edges_to_test.append((rand_user, rand_item))

print("\n--- Testing FAKE Edges ---")
for user, item in fake_edges_to_test:
    score = get_recommendation_score(user, item, embeddings, node_to_int_id)
    print(f"Score for ({user}, {item}): {score:.4f}")


--- Testing REAL Edges ---
Score for (MovieLens_user_5373, MovieLens_item_1197): 6.1909
Score for (MovieLens_user_1862, MovieLens_item_2640): 4.0573
Score for (MovieLens_user_145411, MovieLens_item_235): 5.5962
Score for (MovieLens_user_54133, MovieLens_item_1136): 5.3098
Score for (MovieLens_user_87735, MovieLens_item_3210): 5.0010

--- Testing FAKE Edges ---
Score for (MovieLens_user_22944, MovieLens_item_156352): -0.0579
Score for (MovieLens_user_121725, MovieLens_item_161894): -0.2344
Score for (MovieLens_user_48497, MovieLens_item_88837): 0.2186
Score for (MovieLens_user_106731, MovieLens_item_193253): 0.1148
Score for (MovieLens_user_26934, MovieLens_item_32866): -0.0563


In [29]:
!uv pip freeze


Using Python 3.10.19 environment at: /home/berni/education/Generic-Recommender-Semantics-Put/.venv
aiohappyeyeballs==2.6.1
aiohttp==3.13.2
aiosignal==1.4.0
altair==6.0.0
asttokens==3.0.0
async-timeout==5.0.1
attrs==25.4.0
blinker==1.9.0
cachetools==6.2.2
certifi==2025.11.12
charset-normalizer==3.4.4
click==8.3.1
comm==0.2.3
contourpy==1.3.2
cycler==0.12.1
debugpy==1.8.17
decorator==5.2.1
exceptiongroup==1.3.0
executing==2.2.1
filelock==3.20.0
fonttools==4.60.1
frozenlist==1.8.0
fsspec==2025.10.0
gitdb==4.0.12
gitpython==3.1.45
idna==3.11
ipykernel==7.1.0
ipython==8.37.0
ipywidgets==8.1.8
jedi==0.19.2
jinja2==3.1.6
joblib==1.5.2
jsonschema==4.25.1
jsonschema-specifications==2025.9.1
jupyter-client==8.6.3
jupyter-core==5.9.1
jupyterlab-widgets==3.0.16
kiwisolver==1.4.9
markupsafe==3.0.3
matplotlib==3.10.7
matplotlib-inline==0.2.1
mpmath==1.3.0
multidict==6.7.0
narwhals==2.13.0
nest-asyncio==1.6.0
networkx==3.4.2
numpy==2.2.6
nvidia-cublas-cu12==12.4.5.8
nvidia-cuda-cupti-cu12==12.4.127
n

In [30]:

def evaluate_node2vec(embeddings, test_interactions, train_interactions, node_to_int_id, user_map, item_map, k=20):
    model.eval()

    user_int_indices = [node_to_int_id[uid] for uid in user_map.keys() if uid in node_to_int_id]
    item_int_indices = [node_to_int_id[iid] for iid in item_map.keys() if iid in node_to_int_id]

    user_idx_to_emb_idx = {user_map[uid]: i for i, uid in enumerate(user_map.keys()) if uid in node_to_int_id}
    item_idx_to_emb_idx = {item_map[iid]: i for i, iid in enumerate(item_map.keys()) if iid in node_to_int_id}

    user_embs = embeddings[user_int_indices].to(device)
    item_embs = embeddings[item_int_indices].to(device)

    test_user_indices = test_interactions['user_idx'].unique()

    test_ground_truth = test_interactions.groupby('user_idx')['item_idx'].apply(list).to_dict()
    train_ground_truth = train_interactions.groupby('user_idx')['item_idx'].apply(list).to_dict()

    recalls, precisions, f1_scores = [], [], []

    for user_idx in tqdm(test_user_indices, desc='Evaluating'):
        if user_idx not in user_idx_to_emb_idx: continue

        emb_idx = user_idx_to_emb_idx[user_idx]
        user_emb = user_embs[emb_idx]

        scores = torch.matmul(user_emb, item_embs.T)

        excluded_items = train_ground_truth.get(user_idx, [])
        if excluded_items:
            excluded_emb_indices = [item_idx_to_emb_idx[i] for i in excluded_items if i in item_idx_to_emb_idx]
            if excluded_emb_indices:
                scores[excluded_emb_indices] = -np.inf

        _, top_k_indices = torch.topk(scores, k=k)

        top_k_item_indices = [list(item_idx_to_emb_idx.keys())[i] for i in top_k_indices.cpu().numpy()]

        ground_truth_items = test_ground_truth.get(user_idx, [])
        if not ground_truth_items: continue

        hits = np.isin(top_k_item_indices, ground_truth_items)
        num_hits = np.sum(hits)

        recall = num_hits / len(ground_truth_items)
        precision = num_hits / k
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

        recalls.append(recall)
        precisions.append(precision)
        f1_scores.append(f1)

    avg_recall = np.mean(recalls)
    avg_precision = np.mean(precisions)
    avg_f1 = np.mean(f1_scores)

    print(f'Recall@{k}: {avg_recall:.4f}')
    print(f'Precision@{k}: {avg_precision:.4f}')
    print(f'F1-score@{k}: {avg_f1:.4f}')

    return avg_recall, avg_precision, avg_f1


In [36]:
def evaluate_auc_node2vec(embeddings, test_interactions, all_interactions, node_to_int_id, user_map, item_map, num_neg_samples=100):
    model.eval()

    user_int_indices = [node_to_int_id[uid] for uid in user_map.keys() if uid in node_to_int_id]
    item_int_indices = [node_to_int_id[iid] for iid in item_map.keys() if iid in node_to_int_id]

    user_idx_to_emb_idx = {user_map[uid]: i for i, uid in enumerate(user_map.keys()) if uid in node_to_int_id}
    item_idx_to_emb_idx = {item_map[iid]: i for i, iid in enumerate(item_map.keys()) if iid in node_to_int_id}

    user_embs = embeddings[user_int_indices].to(device)
    item_embs = embeddings[item_int_indices].to(device)

    num_items = len(item_map)

    auc_scores = []
    unique_test_users = test_interactions['user_idx'].unique()

    user_pos_items_all = all_interactions.groupby('user_idx')['item_idx'].apply(set).to_dict()

    for user_idx in tqdm(unique_test_users, desc='Evaluating AUC'):
        if user_idx not in user_idx_to_emb_idx: continue

        pos_items_test = test_interactions[test_interactions['user_idx'] == user_idx]['item_idx'].values
        if len(pos_items_test) == 0: continue

        interacted_items = user_pos_items_all.get(user_idx, set())

        available_neg_items = list(set(range(num_items)) - interacted_items)
        if not available_neg_items: continue

        num_neg_to_sample = min(len(pos_items_test) * num_neg_samples, len(available_neg_items))
        neg_items_sampled = random.sample(available_neg_items, num_neg_to_sample)
        if not neg_items_sampled: continue

        pos_items_test_emb = [item_idx_to_emb_idx[i] for i in pos_items_test if i in item_idx_to_emb_idx]
        neg_items_sampled_emb = [item_idx_to_emb_idx[i] for i in neg_items_sampled if i in item_idx_to_emb_idx]
        if not pos_items_test_emb or not neg_items_sampled_emb: continue

        user_emb = user_embs[user_idx_to_emb_idx[user_idx]]
        pos_scores = torch.sum(user_emb * item_embs[pos_items_test_emb], dim=1)
        neg_scores = torch.sum(user_emb * item_embs[neg_items_sampled_emb], dim=1)

        all_scores = torch.cat([pos_scores, neg_scores]).cpu().detach().numpy()
        labels = np.array([1]*len(pos_scores) + [0]*len(neg_scores))

        if len(np.unique(labels)) < 2: continue

        try:
            auc = roc_auc_score(labels, all_scores)
            auc_scores.append(auc)
        except ValueError:
            pass

    avg_auc = np.mean(auc_scores) if auc_scores else 0
    print(f'Average AUC: {avg_auc:.4f}')
    return avg_auc


In [32]:
evaluate_node2vec(embeddings, test_interactions_df, train_interactions_df, node_to_int_id, user_map, item_map, k=20)


Evaluating:   0%|          | 0/165813 [00:00<?, ?it/s]

Recall@20: 0.2436
Precision@20: 0.1161
F1-score@20: 0.1214


(np.float64(0.2435684924281512),
 np.float64(0.11614770856326102),
 np.float64(0.12136969532888697))

In [37]:
evaluate_auc_node2vec(embeddings, test_interactions_df, all_interactions_df, node_to_int_id, user_map, item_map)


Evaluating AUC:   0%|          | 0/165813 [00:00<?, ?it/s]

Average AUC: 0.9909


np.float64(0.9908723445705948)